In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("employees").getOrCreate()

In [2]:
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("location", StringType(), True),
    StructField("hire_date", DateType(), True),
    StructField("performance_rating", DoubleType(), True),
    StructField("years_exp", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True)
])
df = spark.read \
    .option("header", "true") \
    .option("delimiter", ",") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(schema) \
    .csv("file:///home/itversity/itversity-material/employees.txt")

In [3]:
df.show(truncate=False)

+------+----------------+-----------+-----------------+------+-------------+----------+------------------+---------+------------------------------------------------------------------------------+
|emp_id|name            |department |job_title        |salary|location     |hire_date |performance_rating|years_exp|_corrupt_record                                                               |
+------+----------------+-----------+-----------------+------+-------------+----------+------------------+---------+------------------------------------------------------------------------------+
|1     |John Smith      |Engineering|Senior Developer |125000|San Francisco|2021-03-15|4.5               |8        |null                                                                          |
|2     |Sarah Johnson   |Sales      |Account Executive|85000 |New York     |2022-01-10|4.2               |5        |null                                                                          |
|3     |Michael Will

In [4]:
df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- location: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- performance_rating: double (nullable = true)
 |-- years_exp: integer (nullable = true)
 |-- _corrupt_record: string (nullable = true)



In [5]:
df.filter(col("_corrupt_record").isNotNull()) \
  .show(truncate=False)

+------+----+----------+---------+------+--------+---------+------------------+---------+------------------------------------------------------------------------------+
|emp_id|name|department|job_title|salary|location|hire_date|performance_rating|years_exp|_corrupt_record                                                               |
+------+----+----------+---------+------+--------+---------+------------------+---------+------------------------------------------------------------------------------+
|null  |null|null      |null     |null  |null    |null     |null              |null     |7,Robert Martinez,Legal,Legal Counsel145000,San Francisco,2019-09-22,4.8,10, 5|
+------+----+----------+---------+------+--------+---------+------------------+---------+------------------------------------------------------------------------------+



In [6]:
clean_df = df.filter(col("_corrupt_record").isNull())

In [7]:
clean_df = clean_df.na.drop(how="all")

In [8]:
clean_df.show(truncate=False)

+------+----------------+-----------+-----------------+------+-------------+----------+------------------+---------+---------------+
|emp_id|name            |department |job_title        |salary|location     |hire_date |performance_rating|years_exp|_corrupt_record|
+------+----------------+-----------+-----------------+------+-------------+----------+------------------+---------+---------------+
|1     |John Smith      |Engineering|Senior Developer |125000|San Francisco|2021-03-15|4.5               |8        |null           |
|2     |Sarah Johnson   |Sales      |Account Executive|85000 |New York     |2022-01-10|4.2               |5        |null           |
|3     |Michael Williams|Engineering|Software Engineer|95000 |Austin       |2023-06-20|3.8               |3        |null           |
|4     |Jennifer Brown  |Marketing  |Marketing Manager|92000 |Chicago      |2020-11-05|4.7               |7        |null           |
|5     |David Jones     |Finance    |Senior Analyst   |105000|Boston 

In [9]:
clean_df.groupBy("name") \
        .count() \
        .show()

+----------------+-----+
|            name|count|
+----------------+-----+
|  Jennifer Brown|    1|
|  James Anderson|    1|
|Michael Williams|    1|
|   Sarah Johnson|    1|
| Patricia Wilson|    1|
|      John Smith|    1|
|     David Jones|    1|
|     Lisa Garcia|    1|
|     Mary Thomas|    1|
+----------------+-----+



In [10]:
invalid_df = clean_df.filter(
    col("salary").isNull() |
    col("department").isNull() |
    col("hire_date").isNull()
)

invalid_df.show(truncate=False)

+------+----+----------+---------+------+--------+---------+------------------+---------+---------------+
|emp_id|name|department|job_title|salary|location|hire_date|performance_rating|years_exp|_corrupt_record|
+------+----+----------+---------+------+--------+---------+------------------+---------+---------------+
+------+----+----------+---------+------+--------+---------+------------------+---------+---------------+



In [11]:
valid_df = clean_df.filter(
    col("salary").isNotNull() &
    col("hire_date").isNotNull()
)

In [12]:
valid_df.groupBy("department") \
        .agg(avg("salary").alias("avg_salary")) \
        .show()

+-----------+------------------+
| department|        avg_salary|
+-----------+------------------+
|      Sales|           97500.0|
|Engineering|121666.66666666667|
|         HR|           88000.0|
|    Finance|          105000.0|
|  Marketing|           92000.0|
|         IT|          115000.0|
+-----------+------------------+



In [13]:
valid_df.groupBy("department") \
        .count() \
        .show()

+-----------+-----+
| department|count|
+-----------+-----+
|      Sales|    2|
|Engineering|    3|
|         HR|    1|
|    Finance|    1|
|  Marketing|    1|
|         IT|    1|
+-----------+-----+



In [14]:
valid_df.groupBy("department") \
        .agg(
            avg("salary").alias("avg_salary"),
            avg("years_exp").alias("avg_experience"),
            count("*").alias("employees")
        ) \
        .orderBy(desc("avg_salary")) \
        .show()

+-----------+------------------+-----------------+---------+
| department|        avg_salary|   avg_experience|employees|
+-----------+------------------+-----------------+---------+
|Engineering|121666.66666666667|7.666666666666667|        3|
|         IT|          115000.0|              4.0|        1|
|    Finance|          105000.0|              6.0|        1|
|      Sales|           97500.0|              6.0|        2|
|  Marketing|           92000.0|              7.0|        1|
|         HR|           88000.0|              6.0|        1|
+-----------+------------------+-----------------+---------+

